In [1]:
#get all the tables:
from db_setup import conn, currencyexchange, customer, date, product, sales, store
#instead of saving csv files in sql and then importing one by one
import pandas as pd


c:\Users\amyre\Desktop\Luke Barousse SQL Course\SQL + Contoso\db_setup.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  currencyexchange = pd.read_sql("SELECT * FROM currencyexchange", conn)
c:\Users\amyre\Desktop\Luke Barousse SQL Course\SQL + Contoso\db_setup.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  customer = pd.read_sql("SELECT * FROM customer", conn)
c:\Users\amyre\Desktop\Luke Barousse SQL Course\SQL + Contoso\db_setup.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  date = pd.read_sql("SELECT * FR

0- Cleanup our dates

In [2]:
#checked with df.info() and the date columns are strings. do converting all of them (even those i havent checked):

sales["orderdate"] = pd.to_datetime(sales["orderdate"])
sales["deliverydate"] = pd.to_datetime(sales["deliverydate"])
customer["birthday"] = pd.to_datetime(customer["birthday"])
customer["startdt"] = pd.to_datetime(customer["startdt"])
customer["enddt"] = pd.to_datetime(customer["enddt"])
store["opendate"] = pd.to_datetime(store["opendate"])
store["closedate"] = pd.to_datetime(store["closedate"])
date["date"] = pd.to_datetime(date["date"])
currencyexchange["date"] = pd.to_datetime(currencyexchange["date"])



In [3]:
#can confirm each table using the .info(), but copilot gave a faster way to do it:

tables = {
    "sales": sales,
    "customer": customer,
    "store": store,
    "date": date,
    "currencyexchange": currencyexchange,
    "product": product
}

for name, df in tables.items():
    print(f"Table: {name}")
    print(df.info())
    print("\n")

Table: sales
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 199873 entries, 0 to 199872
Data columns (total 13 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   orderkey      199873 non-null  int64         
 1   linenumber    199873 non-null  int64         
 2   orderdate     199873 non-null  datetime64[ns]
 3   deliverydate  199873 non-null  datetime64[ns]
 4   customerkey   199873 non-null  int64         
 5   storekey      199873 non-null  int64         
 6   productkey    199873 non-null  int64         
 7   quantity      199873 non-null  int64         
 8   unitprice     199873 non-null  float64       
 9   netprice      199873 non-null  float64       
 10  unitcost      199873 non-null  float64       
 11  currencycode  199873 non-null  object        
 12  exchangerate  199873 non-null  float64       
dtypes: datetime64[ns](2), float64(4), int64(6), object(1)
memory usage: 19.8+ MB
None


Table: customer
<class

1.Row count

In [ ]:

row_counts=pd.DataFrame({
    "table_name":tables.keys(),
    "row_count":[len(df) for df in tables.values()]
})


row_counts.sort_values(by="row_count", ascending=False)

,table_name,row_count
0,sales,199873
1,customer,104990
4,currencyexchange,91325
3,date,3653
5,product,2517
2,store,74


2 - check sales table:
info like rows, unique orders/customers/products/stores

In [6]:
sales_info=pd.DataFrame({
    "sales_rows": [len(sales)],
    "unique_orders": [sales["orderkey"].nunique()],
    "unique_customers": [sales["customerkey"].nunique()],
    "unique_products": [sales["productkey"].nunique()],
    "unique_stores": [sales["storekey"].nunique()]
})

print(sales_info)

   sales_rows  unique_orders  unique_customers  unique_products  unique_stores
0      199873          83130             49487             2517             72


3- check wether orders appear more than once

In [ ]:
#Lets count how many lines per order:
order_line_count=sales.groupby("orderkey").size().reset_index(name="line_count")
#number of lines per order

total_order_lines=order_line_count["line_count"].sum()
total_orders=len(order_line_count)
multi_line_orders=(order_line_count["line_count"]>1).sum()
avg_lines_per_order=round(order_line_count["line_count"].mean(),2)
#or avg_lines_per_order=total_order_lines/total_orders
max_lines=order_line_count["line_count"].max()
min_lines=order_line_count["line_count"].min()
pct_multi_line=round(multi_line_orders/total_orders,2)

summary = pd.DataFrame([{
    "total_order_lines": total_order_lines,
    "total_orders": total_orders,
    "multi_line_orders": multi_line_orders,
    "avg_lines_per_order": avg_lines_per_order,
    "min_lines": min_lines,
    "max_lines": max_lines,
    "pct_multi_line": pct_multi_line
}])

print(summary)



   total_order_lines  total_orders  multi_line_orders  avg_lines_per_order  \
0             199873         83130              54171                  2.4   

   min_lines  max_lines  pct_multi_line  
0          1          7            0.65  


4-Null checks

In [ ]:
sales.isnull().sum().to_frame("missing values")


,missing values
orderkey,0
linenumber,0
orderdate,0
deliverydate,0
customerkey,0
storekey,0
productkey,0
quantity,0
unitprice,0
netprice,0


5- Check unusual values

In [10]:
pd.DataFrame([{
    "nonpositive_quantity": (sales["quantity"] <= 0).sum(),
    "nonpositive_unitprice": (sales["unitprice"] <= 0).sum(),
    "nonpositive_netprice": (sales["netprice"] <= 0).sum(),
    "negative_unitcost": (sales["unitcost"] < 0).sum(),
    "nonpositive_exchangerate": (sales["exchangerate"] <= 0).sum()
}])

,nonpositive_quantity,nonpositive_unitprice,nonpositive_netprice,negative_unitcost,nonpositive_exchangerate
0,0,0,0,0,0


6-Revenue, cost, profit

In [33]:
sales["line_revenue_usd"]=(sales["quantity"]*sales["netprice"]/sales["exchangerate"]).round(2)
sales["line_cost_usd"]=(sales["quantity"]*sales["unitcost"]/sales["exchangerate"]).round(2)
sales["line_profit_usd"]=(sales["line_revenue_usd"]-sales["line_cost_usd"]).round(2)

7- Date ranges:

In [11]:
pd.DataFrame([{
    "min_orderdate":sales["orderdate"].min(),
    "max_orderdate":sales["orderdate"].max(),
    "min_deliverydate":sales["deliverydate"].min(),
    "max_deliverydate":sales["deliverydate"].max()
}])

,min_orderdate,max_orderdate,min_deliverydate,max_deliverydate
0,2015-01-01,2024-04-20,2015-01-01,2024-04-27


8- Delivery log check:

In [41]:
sales["delivery_days"]=(sales["deliverydate"]-sales["orderdate"]).dt.days

pd.DataFrame([{
    "min_delivery_days":sales["delivery_days"].min(),
    "max_delivery_days":sales["delivery_days"].max(),
    "avg_delivery_days":sales["delivery_days"].mean(),
    "impossible_delivery_log":(sales["delivery_days"]<0).sum()
}])

,min_delivery_days,max_delivery_days,avg_delivery_days,impossible_delivery_log
0,0,19,1.30678,0


9 - confirm if unitprice and netprice differ (discounts are true?)

In [ ]:
pd.DataFrame([{
    "total_rows":len(sales),
    "same_price_rows":(sales["unitprice"]==sales["netprice"]).sum(),
    "different_price_rows":(sales["unitprice"]!=sales["netprice"]).sum(),
    "pct_different_price_rows":(sales["unitprice"]!=sales["netprice"]).mean()
    #the mean of boolean values is pretty much the percentage
}])

,total_rows,same_price_rows,different_price_rows,pct_different_price_rows
0,199873,77533,122340,0.612089


In [48]:
(sales["unitprice"]==sales["netprice"]).size

199873

10 - check duplicate keys in the customer and product tables:

In [ ]:
print(f'customerkey duplicates:',customer["customerkey"].duplicated().sum())
print(f'productkey duplicates:',product["productkey"].duplicated().sum())



customerkey duplicates: 0
productkey duplicates: 0


11 - check if there are customers in sales that dont exist in customer

In [ ]:
(sales[["customerkey"]].drop_duplicates().merge(customer[["customerkey"]],on="customerkey",how="left",indicator=True)["_merge"]=="left_only").sum()

np.int64(0)

12 - check duplicate order lines in sales table:

In [ ]:
sales.groupby(["orderkey","linenumber"]).size().reset_index(name="instances").query("instances>1")



,orderkey,linenumber,instances
